# Person 3: sections 6 (speedup and Amdahl's law) and 7 (non-determinism)

Paste these cells into the main notebook after section 5, before section 8. The first code cell only exists so this notebook runs on its
own: in the merged notebook the same variables already come from sections 1 to 5, so leave its imports and drop the rest.
All settings (worker counts, repeats, iterations, shards) are read from `config.py`. Run Part 1 first: it writes
`results/demand_zone_hour.parquet`.

In [ ]:
import config                      # first: it pins numpy to one thread per process
import numpy as np
import pandas as pd

from src import cleaning
from src.benchmark import (speedup_table, plot_speedup, time_it, start_empty_pool, round_trip_seconds,
                           payload_mb)
from src.features import build_features, month_rows, to_arrays, fit_scaler, apply_scaler, fold_boundaries
from src.model import make_shards, shard_gradient, take_step, train_parallel, search_sequential, search_parallel

# Standalone only: the merged notebook already has these from sections 1-5.
cleaning_shards = cleaning.list_shards()
dev = month_rows(build_features(pd.read_parquet(config.RESULTS_DIR / "demand_zone_hour.parquet")),
                 config.DEV_MONTHS)
X_dev_raw, y_dev, zones_dev = to_arrays(dev)
X_dev = apply_scaler(X_dev_raw, *fit_scaler(X_dev_raw))
folds = fold_boundaries(dev["hour"].to_numpy(), config.VAL_FOLD_STARTS, config.VAL_FOLD_DAYS)
grad_shards = make_shards(X_dev, y_dev, zones_dev)
search_data = {"X": X_dev_raw, "y": y_dev}
LR, L2, N_ITER = config.DEFAULT_LR, config.DEFAULT_L2, config.GD_ITERATIONS

print(f"Workers {config.WORKER_COUNTS}, {config.N_REPEATS} timed runs per setting after 1 untimed warm-up")

## 6. Speedup and Amdahl's law

We time the same three jobs one after another (serial) and in parallel:

| Job | Kind of parallelism | Serial version | Parallel version |
|---|---|---|---|
| Cleaning (section 2) | data parallel, 16 row-group shards | `build_demand_sequential` | `build_demand_parallel` |
| Gradient (section 4) | task parallel, 6 zone shards, 300 iterations | the same 6 shard sums, in a loop | `train_parallel` |
| Grid search (section 5) | task parallel, 9 independent trials | `search_sequential` | `search_parallel` |

**How each number is measured** (`speedup_table()` in `src/benchmark.py`):
- **Warm-up:** each version runs once without being timed. That first run pays one-off costs (reading the files from disk into
  memory, loading libraries), which would otherwise land on whichever version happened to go first.
- **Repeats:** then it runs `N_REPEATS` more times and we report the mean and standard deviation. One timing can be thrown off by
  anything else the machine is doing; the spread shows how much to trust the mean.
- **Same work, same answer:** every parallel result is checked to be *exactly* equal to the serial one, or the cell stops.
- **The baseline is the plain serial function, not a pool with one worker.** The gap between those two rows is the cost of
  starting a worker process and copying the data to it.

**Speedup** S(n) = serial time / time with n workers. **Efficiency** E(n) = S(n) / n, the share of each worker we actually used
(1.0 is perfect). **Amdahl's law** says S(n) = 1 / ((1 - p) + p / n), where p is the fraction of the work that can run in parallel.
Solving for p gives the `amdahl_p` column: **p = (1 - 1/S) / (1 - 1/n)**. The best possible speedup with any number of workers is 1 / (1 - p).
Amdahl's law assumes parallel work costs nothing extra, so when parallel is *slower* than serial (S < 1) p comes out negative: that
means the overheads cost more than the parallel part saves.

Every setting runs 4 times, so this section takes several minutes.

### 6.1 Cleaning (data parallelism)

In [ ]:
cleaning_table = speedup_table(cleaning.build_demand_sequential, cleaning.build_demand_parallel, cleaning_shards)
cleaning_table.to_csv(config.RESULTS_DIR / "speedup_cleaning.csv", index=False)
cleaning_table.round(3)

### 6.2 Gradient computation (task parallelism)

The serial version here adds up the **same 6 shard sums in the same order** as `train_parallel`, just without worker processes.
Section 4's `train_sequential` sums all rows in one go instead, and because floating-point addition depends on order
((a + b) + c is not always a + (b + c) in the last digit), its weights differ from the parallel ones by about 1e-14. With the
same order the two are identical bit for bit, so serial and parallel really do the same work.

In [ ]:
def train_sharded_sequential(shards, lr, l2, n_iter):
    """Serial twin of train_parallel: same shards, same shard-order sum, no worker processes."""
    # shortcut: defined here until it is added to src/model.py next to train_parallel
    n_total = sum(len(y) for _, y in shards)
    w, b = np.zeros(shards[0][0].shape[1]), 0.0
    for _ in range(n_iter):
        gw, gb = np.zeros_like(w), 0.0
        for X, y in shards:
            part_gw, part_gb, _ = shard_gradient(w, b, X, y)
            gw, gb = gw + part_gw, gb + part_gb
        w, b = take_step(w, b, gw / n_total, gb / n_total, lr, l2)
    return w, b


gradient_table = speedup_table(lambda shards: train_sharded_sequential(shards, LR, L2, N_ITER),
                               lambda shards, workers: train_parallel(shards, LR, L2, N_ITER, workers),
                               grad_shards)
gradient_table.to_csv(config.RESULTS_DIR / "speedup_gradient.csv", index=False)
gradient_table.round(3)

### 6.3 Hyperparameter search (task parallelism)

Each trial records how long it took (`seconds`), which is different on every run by nature, so that one field is left out when
the serial and parallel results are compared. The learning rate, L2 strength and validation MAE of all 9 trials must match exactly.

In [ ]:
def without_timing(results):
    return [{k: v for k, v in r.items() if k != "seconds"} for r in results]


search_table = speedup_table(lambda data, folds: without_timing(search_sequential(data, folds, N_ITER)),
                             lambda data, folds, workers: without_timing(search_parallel(data, folds, N_ITER, workers)),
                             search_data, folds)
search_table.to_csv(config.RESULTS_DIR / "speedup_search.csv", index=False)
search_table.round(3)

### 6.4 All three jobs together

In [ ]:
speedup_tables = {"cleaning (data parallel)": cleaning_table,
                  "gradient (task parallel)": gradient_table,
                  "grid search (task parallel)": search_table}
summary = pd.concat({name: t.set_index(["version", "workers"])[["mean_s", "std_s", "speedup", "efficiency", "amdahl_p"]]
                     for name, t in speedup_tables.items()})
print(summary.round(3).to_string())
plot_speedup(speedup_tables)

### 6.5 Measuring the overheads

Amdahl's law only explains the part of the work that stays serial. The rest of the gap to ideal speedup is **overhead**: extra
work that only exists because we went parallel. We measure the three biggest ones directly:

- **Process start-up:** start a pool of empty workers and shut it down again, with no data and no work.
- **Communication per round trip:** on an already-running pool, send 6 empty tasks and wait for the answers. The parallel gradient
  does this once per iteration (300 times), to send `w` out and get the shard sums back.
- **Data transfer:** how many MB each worker receives when it starts (every worker gets a copy of all the shards or all the search data).

In [ ]:
overheads = []
for n in config.WORKER_COUNTS:
    _, startup_times = time_it(start_empty_pool, n)
    overheads.append({"workers": n, "empty pool start-up (s)": np.mean(startup_times),
                      "round trip (ms)": 1000 * round_trip_seconds(n)})
overheads = pd.DataFrame(overheads)
overheads.to_csv(config.RESULTS_DIR / "overheads.csv", index=False)

print(f"Data copied into each worker at start-up: gradient shards {payload_mb(grad_shards):.0f} MB, "
      f"search data {payload_mb((search_data, folds, N_ITER)):.0f} MB")
print(f"Gradient: {N_ITER} iterations x {len(grad_shards)} tasks = {N_ITER * len(grad_shards):,} tasks sent to workers")
overheads.round(3)

**Where the parallel gradient's time goes.** If parallel were free, the time would be the serial time divided by the
number of workers. The table adds the overheads we measured on top of that, and whatever is left over is the overhead we
did not measure separately: copying the shards into every worker, and each iteration waiting for the slowest shard. A negative leftover means the serial run was slower than the same work inside a worker: machine noise, not a real saving (see the check in 6.6).

In [ ]:
serial_s = gradient_table.mean_s[0]
breakdown = pd.DataFrame({
    "workers": overheads["workers"],
    "measured parallel (s)": [gradient_table.query("version == 'parallel' and workers == @n").mean_s.item()
                              for n in overheads["workers"]],
    "real work / n (s)": serial_s / overheads["workers"],
    "process start-up (s)": overheads["empty pool start-up (s)"],
    f"{N_ITER} round trips (s)": overheads["round trip (ms)"] * N_ITER / 1000,
})
breakdown["rest: copying data, waiting (s)"] = breakdown["measured parallel (s)"] - breakdown.iloc[:, 2:5].sum(axis=1)
breakdown.to_csv(config.RESULTS_DIR / "gradient_time_breakdown.csv", index=False)
breakdown.round(3)

### 6.6 Why each job speeds up (or doesn't)

**Gradient: little or no speedup.** One iteration of real work is only a few milliseconds. In the breakdown above only one part
gets smaller as workers are added: the real work, divided over the workers. Everything else is overhead that stays the same or
grows with more workers: starting the processes, copying about 60 MB of shards into *each* worker, and 300 round trips (every
iteration sends `w` out and waits for all 6 shard sums to come back, so it also waits for the slowest one). With this little work
per iteration the overheads eat most of the gain, so speedup stays near 1, or below 1 when they outweigh it. That is also why
`amdahl_p` comes out small or negative: what limits this job is overhead, not a serial part of the code. More work per iteration
(more rows, a bigger model) would help; more workers would not.

**Grid search: the best speedup.** Each trial is several seconds of independent work (4 folds x 300 iterations), and a worker
only talks to the main process twice per trial: once to receive its settings, once to return its score. Start-up and data copying
are paid once and then hidden behind a lot of real work, so speedup grows with workers. It should still fall short of ideal,
because of that start-up and copying and because of **load imbalance**: 9 trials split evenly over 3 workers (3 rounds), but over
2 workers they take 5 rounds, the last one with a worker idle.

**Cleaning: in between.** Each shard is a big piece of work (a row group of about a million trips) and sends back only a small
table of counts, so communication is cheap. What limits it is reading the files (all workers share the same disk), starting the
processes, and the merge at the end, which can only run once every shard is done: the serial part in Amdahl's law.

**Check before trusting a run.** Efficiency above 1 (or p above 1) means parallel beat the ideal, which is very unlikely for these
jobs. It means the serial run was slowed down by something else: on a laptop, Windows moving the process to a slower core, or
another program. The large `std_s` on those rows shows the same thing. The final numbers come from Rockhill, where a job gets its
own cores.

## 7. Non-determinism: same code, same data, different answer

A parallel program is **non-deterministic** when running it twice on the same data can give different results. Here we
show three real sources in our own pipeline. For each one: two or more runs that differ, the fix, a re-run showing the
output is now identical (checked with `assert`), and what the fix does *not* cover.

| Source | Where it shows up | Fix |
|---|---|---|
| 7.1 Completion order | cleaning: shards finish in a different order each run | sort results by shard id before merging |
| 7.2 Floating-point sum order | gradient: shard sums added in a different order | always add them in shard order |
| 7.3 Thread count | numpy's maths library splits sums across threads | pin it to 1 thread per process (`config.py`) |

This section takes under a minute: none of the demos are timed.

### 7.1 Completion order

`collect_in_completion_order()` (in `src/cleaning.py`) collects the shard results with `as_completed`: in whatever order the
workers *finish*, which depends on how busy each core is at that moment. We run it three times and look at the order.

In [ ]:
order_runs = [cleaning.collect_in_completion_order(cleaning_shards, config.MAX_WORKERS) for _ in range(3)]
for i, run in enumerate(order_runs, 1):
    print(f"run {i} finished the shards in this order: {[shard_id for shard_id, _, _ in run]}")

# Before the fix: glue the partial count tables together in the order they arrived.
glued = [pd.concat([counts for _, counts, _ in run], ignore_index=True) for run in order_runs]
order_differed = not all(g.equals(glued[0]) for g in glued)
print("\nBefore the fix, the glued tables differ between runs:", order_differed)

# The fix: merge_partials() sorts by shard id first, then adds up the counts.
merged = [cleaning.merge_partials(run)[0] for run in order_runs]
order_fixed = all(m.equals(merged[0]) for m in merged)
print("After the fix (sort by shard id, then merge), all runs identical:", order_fixed)
assert order_fixed

**Fix and its limits.** Each result carries its shard id, and `merge_partials()` sorts by it before combining, so the
order the workers finish in no longer matters. `build_demand_parallel()` gets the same effect from `pool.map`, which returns
results in the order the shards were sent. The limits: the ids must be fixed before the work starts (here: file, then row
group), so a different sharding (another file layout, or `iter_batches` with another batch size) still gives a different
order, just a repeatable one. And sorting means waiting for *every* shard before merging, so results cannot be used as
they stream in. If all three runs above happened to finish in the same order, run the cell again: the point is that the
order is not guaranteed.

### 7.2 Floating-point sum order

Computers store numbers with about 16 significant digits, so every addition rounds a little, and the rounding depends on
the order: `(a + b) + c` and `a + (b + c)` can differ in the last digit. The parallel gradient adds up 6 shard sums every
iteration. First: how many different answers do the same 6 numbers give, over every order they could arrive in?

In [ ]:
from itertools import permutations

w0, b0 = np.zeros(X_dev.shape[1]), 0.0
shard_sums = [shard_gradient(w0, b0, X, y)[0] for X, y in grad_shards]   # the 6 shard sums of one iteration


def add_in_order(order):
    total = np.zeros_like(shard_sums[0])
    for i in order:
        total = total + shard_sums[i]
    return total


all_orders = [add_in_order(order) for order in permutations(range(len(shard_sums)))]
print(f"The same {len(shard_sums)} shard sums added in all {len(all_orders)} possible orders give "
      f"{len({s.tobytes() for s in all_orders})} different answers")
print(f"Largest difference from shard order: {max(np.abs(s - all_orders[0]).max() for s in all_orders):.1e}, "
      f"on numbers of size up to {np.abs(all_orders[0]).max():.1e}")

One gradient is off only in the last digits, but training repeats this 300 times and every step starts from the last
one, so the differences carry through to the final model. Below, two trainings add the shard sums in **arrival order**
(a random order stands in for whichever worker finishes first, as with `as_completed`), then two add them in
**shard order**, which is what `train_parallel` does: `pool.map` returns the results in shard order.

In [ ]:
def train_arrival_order(shards, rng):
    """train_sharded_sequential, but each iteration adds the shard sums in a random order: whichever finished first."""
    n_total = sum(len(y) for _, y in shards)
    w, b = np.zeros(shards[0][0].shape[1]), 0.0
    for _ in range(N_ITER):
        sums = [shard_gradient(w, b, X, y) for X, y in shards]
        gw, gb = np.zeros_like(w), 0.0
        for i in rng.permutation(len(shards)):
            gw, gb = gw + sums[i][0], gb + sums[i][1]
        w, b = take_step(w, b, gw / n_total, gb / n_total, LR, L2)
    return w, b


# Unseeded on purpose: each run gets its own arrival orders, like real workers would.
(w_a, b_a), (w_b, b_b) = (train_arrival_order(grad_shards, np.random.default_rng()) for _ in range(2))
sum_order_differed = not (np.array_equal(w_a, w_b) and b_a == b_b)
print(f"Before the fix, two trainings in arrival order differ: {sum_order_differed} "
      f"(largest weight difference {np.abs(w_a - w_b).max():.1e})")

(w_c, b_c), (w_d, b_d) = (train_sharded_sequential(grad_shards, LR, L2, N_ITER) for _ in range(2))
sum_order_fixed = np.array_equal(w_c, w_d) and b_c == b_d
print("After the fix (always shard order), two trainings identical:", sum_order_fixed)
assert sum_order_fixed

**Fix and its limits.** Adding in a fixed order (shard 0, then 1, ..., then 5) makes every run identical, and
`train_parallel` gets it for free because `pool.map` keeps the shard order. It makes the answer *repeatable*, not *more
accurate*: shard order is no more correct than any other order. It also only holds while the shards stay the same. With a
different `N_GRADIENT_SHARDS` the partial sums change, and so do the last digits. That is why the number of shards is fixed
in `config.py` and does not change with the number of workers, so 1, 2 or 3 workers all give the same model. For the same
reason, section 4's `train_sequential`, which adds all rows in one go, differs from the parallel version by about 1e-14.

### 7.3 Thread count

numpy hands matrix maths to a library (OpenBLAS) that can split one big sum across several threads and then add their
partial results together: the same rounding problem as 7.2, inside one line of numpy. `config.py` pins it to 1 thread.
Here we undo the pin for a moment (`threadpool_limits`) and compute the same gradient with different numbers of threads.
Each thread count is run twice.

In [ ]:
import os
from threadpoolctl import threadpool_info, threadpool_limits

w_test = np.linspace(-1, 1, X_dev.shape[1])            # any fixed weights will do
cores = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
thread_rows = []
for n_threads in sorted({1, 2, 4, cores}):
    with threadpool_limits(n_threads):                 # undo config.py's pin, only inside this block
        first = X_dev.T @ (X_dev @ w_test - y_dev)
        second = X_dev.T @ (X_dev @ w_test - y_dev)
    if n_threads == 1:
        one_thread = first
    thread_rows.append({"threads": n_threads, "same answer twice": np.array_equal(first, second),
                        "same as 1 thread": np.array_equal(first, one_thread),
                        "largest difference from 1 thread": np.abs(first - one_thread).max()})
thread_table = pd.DataFrame(thread_rows)
print(thread_table.to_string(index=False))

threads_differed = not thread_table["same as 1 thread"].all()
print("\nBefore the fix, some thread counts give a different answer:", threads_differed)
pinned = [lib["num_threads"] for lib in threadpool_info()]
threads_fixed = all(n == 1 for n in pinned)
print("After the fix (config.py), threads used by numpy's maths library:", pinned)
assert threads_fixed

**Fix and its limits.** `config.py` sets `OMP_NUM_THREADS`, `OPENBLAS_NUM_THREADS` and the other thread settings to 1
*before* numpy loads (`src/__init__.py` imports it first, so this holds however the code is started), and every worker
process inherits the setting. One thread means one fixed order of additions, on any machine. The answer can depend on the
exact thread count, not just "one or many", which is why it is pinned to exactly 1 rather than to "however many cores
there are". The limits: it only works for libraries that read these settings. A different numpy or OpenBLAS **version** can
still change the last digits, which is why `requirements.txt` pins versions. And each process gives up multi-threaded maths,
which costs nothing here because the parallelism comes from worker processes (3 workers each running a full set of threads
would overload the machine anyway).

### 7.4 Randomness: none, by design

Unseeded random numbers are the most common source of non-determinism, but this pipeline uses none: the model starts from
`w = 0`, the grid search tries a fixed list of 9 settings, the folds are fixed dates and the shards are fixed zone groups. The
one random draw in this notebook is the deliberate "arrival order" demo in 7.2. If random search or random start weights were
added, each task would get its own seed from `config.SEED` and its task id, never from the worker or process id
(`np.random.default_rng([config.SEED, task_id])`), so the result would not depend on which worker ran which task.

### 7.5 Summary

In [ ]:
nondeterminism = pd.DataFrame([
    {"source": "completion order", "differed before the fix": order_differed,
     "fix": "sort by shard id before merging", "identical after the fix": order_fixed},
    {"source": "floating-point sum order", "differed before the fix": sum_order_differed,
     "fix": "always add shard sums in shard order", "identical after the fix": sum_order_fixed},
    {"source": "thread count", "differed before the fix": threads_differed,
     "fix": "pin numpy to 1 thread (config.py)", "identical after the fix": threads_fixed},
])
nondeterminism.to_csv(config.RESULTS_DIR / "nondeterminism.csv", index=False)
nondeterminism